In [44]:
!pip -q install azure-ai-documentintelligence azure-core
!pip install azure-storage-blob

In [45]:
"""
Batch OCR of Amazon invoices (MAIN DETAILS ONLY)
using Azure AI Document Intelligence (Prebuilt Read)
"""

from azure.core.credentials import AzureKeyCredential
from azure.ai.documentintelligence import DocumentIntelligenceClient
from azure.ai.documentintelligence.models import AnalyzeDocumentRequest
from azure.storage.blob import ContainerClient
from openpyxl import Workbook
import re
import socket

# -------------------------
# Startup / Secrets
# -------------------------
print("Starting...")
socket.gethostbyname("canadatest.cognitiveservices.azure.com")

from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

endpoint = user_secrets.get_secret("AZURE_DOCINT_ENDPOINT")
key = user_secrets.get_secret("AZURE_DOCINT_KEY")
container_sas_url = user_secrets.get_secret("AZURE_BLOB_CONTAINER_SAS_URL")

print("Endpoint loaded:", bool(endpoint))
print("Key loaded:", bool(key))
print("Container SAS loaded:", bool(container_sas_url))

# -------------------------
# Azure Clients
# -------------------------
docint_client = DocumentIntelligenceClient(
    endpoint=endpoint,
    credential=AzureKeyCredential(key)
)

container_client = ContainerClient.from_container_url(container_sas_url)

def extract_invoice_fields(text: str) -> dict:
    fields = {
        "order_number": None,
        "order_date": None,
        "grand_total": None
    }

    # Matches both: "Details for Order #..." and "Order #..."
    m = re.search(r"(?:Details\s+for\s+Order|Order)\s*#\s*(\d{3}-\d{7}-\d{7})", text)
    if m:
        fields["order_number"] = m.group(1)

    m = re.search(r"Order Placed:\s*([A-Za-z]+\s+\d{1,2},\s+\d{4})", text)
    if m:
        fields["order_date"] = m.group(1)

    # Works for your test PDFs; if some docs say "Grand Total", you can add a fallback
    m = re.search(r"Order Total:\s*\$([\d]+\.\d{2})", text)
    if m:
        fields["grand_total"] = float(m.group(1))

    return fields


def analyze_all_documents(sheet):
    base_url, sas = container_sas_url.split("?", 1)
    seen = set()

    for blob in container_client.list_blobs(name_starts_with="Amazon/"):

        if not blob.name.lower().endswith((".pdf", ".png", ".jpg", ".jpeg")):
            continue

        # prevent duplicates in the same run
        if blob.name in seen:
            continue
        seen.add(blob.name)

        blob_url = f"{base_url}/{blob.name}?{sas}"
        print(f"Processing: {blob.name}")

        poller = docint_client.begin_analyze_document(
            "prebuilt-read",
            AnalyzeDocumentRequest(url_source=blob_url)
        )

        result = poller.result()
        text = result.content or ""

        # fields MUST be created before use
        fields = extract_invoice_fields(text)

        sheet.append([
            blob.name,
            fields["order_number"],
            fields["order_date"],
            fields["grand_total"]
        ])


# Entry Point
if __name__ == "__main__":

    # 🔹 CREATE A NEW EXCEL FILE EVERY RUN
    from openpyxl import Workbook

    workbook = Workbook()
    sheet = workbook.active
    sheet.title = "Amazon Invoices"

    sheet.append([
        "source_file",
        "order_number",
        "order_date",
        "grand_total"
    ])

    analyze_all_documents(sheet)

    output_path = "/kaggle/working/amazon_invoice_main_details.xlsx"
    workbook.save(output_path)

    print(f"\n✅ Fresh Excel file written: {output_path}")

Starting...
Endpoint loaded: True
Key loaded: True
Container SAS loaded: True
Processing: Amazon/Amazon01.pdf
Processing: Amazon/Amazon02.pdf
Processing: Amazon/Amazon03.pdf
Processing: Amazon/Amazon04.pdf
Processing: Amazon/Amazon_Invoice_1.pdf
Processing: Amazon/Amazon_Invoice_2.pdf
Processing: Amazon/Amazon_Invoice_3.pdf

✅ Fresh Excel file written: /kaggle/working/amazon_invoice_main_details.xlsx
